In [0]:
display(spark.sql(
    """
    SELECT 
        MIN(transaction_date) AS min_date,
        MAX(transaction_date) AS max_date
    FROM fraud_detection.bronze.transactions_raw
    """
))

In [0]:
import requests

start_date = "2024-01-01"
end_date = "2024-04-03"

resp = requests.get(f"https://api.frankfurter.dev/v1/{start_date}..{end_date}?base=USD")
resp.raise_for_status()
data = resp.json()

print(f"Number of dates returned: {len(data['rates'])}")
# peek at one day's structure
sample_date = list(data['rates'].keys())[0]
print(sample_date, data['rates'][sample_date])

In [0]:
rows = []
for date, currency_rates in data['rates'].items():
    for currency, rate in currency_rates.items():
        rows.append({"rate_date": date, "currency": currency, "rate_to_usd": float(rate)})

fx_df = spark.createDataFrame(rows)
display(fx_df.limit(10))
print(f"Total rows: {fx_df.count()}")

In [0]:
print(f"Total rows: {fx_df.count()}")

fx_df.write.mode("overwrite").saveAsTable("fraud_detection.bronze.fx_rates_raw")

display(spark.sql("SELECT COUNT(*) FROM fraud_detection.bronze.fx_rates_raw"))